## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 2: Collaborative Filtering
## Task 4: Item-Based Collaborative Filtering

Recommend items based on their similarity to items a user has previously liked or rated highly.
Key idea: if two items are rated similarly by many users, they are likely similar.

**Dataset:** MovieLens ml-latest-small

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'scikit-learn', 'numpy', 'scipy', '-q'])

import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
print('All imports successful!')

All imports successful!


### Step 1: Load Dataset and Build Rating Matrix

In [2]:
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
zip_path = "ml-latest-small.zip"
extract_dir = "ml-latest-small"

if not os.path.exists(extract_dir):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')
    print("Downloaded and extracted.")
else:
    print("Dataset already exists.")

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))
print(f"Movies: {len(movies)}, Ratings: {len(ratings)}, Users: {ratings['userId'].nunique()}")

Dataset already exists.
Movies: 9742, Ratings: 100836, Users: 610


In [3]:
# User-movie rating matrix
rating_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')
print(f"Rating matrix shape: {rating_matrix.shape}")
print(f"Sparsity: {rating_matrix.isna().sum().sum() / (rating_matrix.shape[0] * rating_matrix.shape[1]) * 100:.2f}%")

Rating matrix shape: (610, 9724)
Sparsity: 98.30%


### Step 2: Compute Item-Item Similarity (Pearson Correlation)

We compute Pearson correlation between items (columns) based on how users rated them. This captures co-rating patterns across users.

In [4]:
# Compute item-item Pearson correlation
# corr() on the rating matrix directly gives column-column (item-item) correlation
item_similarity = rating_matrix.corr(method='pearson', min_periods=5)
print(f"Item similarity matrix shape: {item_similarity.shape}")
item_similarity.iloc[:5, :5]

Item similarity matrix shape: (9724, 9724)


movieId,1,2,3,4,5
movieId,,,,,
1,1.000000,0.330978,0.487109,NaN,0.310971
2,0.330978,1.000000,0.419564,NaN,0.562791
3,0.487109,0.419564,1.000000,NaN,0.602266
4,NaN,NaN,NaN,1.0,NaN
5,0.310971,0.562791,0.602266,NaN,1.000000


### Step 3: Predict Ratings Based on Similar Items

For a user $u$ and unrated movie $m$, the predicted rating is a weighted average of the user's ratings on similar items:

$$\hat{r}_{u,m} = \frac{\sum_{j \in \text{rated}} \text{sim}(m, j) \cdot r_{u,j}}{\sum_{j \in \text{rated}} |\text{sim}(m, j)|}$$

Higher similarity = higher weight in the prediction.

In [5]:
def predict_rating_item_cf(user_id, movie_id, rating_matrix, item_similarity, K=30):
    """
    Predict rating for user-movie pair using K most similar items
    that the user has already rated.
    """
    if movie_id not in item_similarity.columns:
        return np.nan
    
    # Items the user has rated
    user_ratings = rating_matrix.loc[user_id].dropna()
    
    # Similarities between target movie and user's rated movies
    rated_items = user_ratings.index.intersection(item_similarity.columns)
    sims = item_similarity.loc[movie_id, rated_items].dropna()
    
    if sims.empty:
        return np.nan
    
    # Select top-K most similar items
    top_k = sims.abs().nlargest(K).index
    top_sims = sims[top_k]
    top_ratings = user_ratings[top_k]
    
    # Filter zero similarities
    mask = top_sims != 0
    top_sims = top_sims[mask]
    top_ratings = top_ratings[mask]
    
    if top_sims.empty:
        return np.nan
    
    # Weighted average
    numerator = (top_sims * top_ratings).sum()
    denominator = top_sims.abs().sum()
    
    prediction = numerator / denominator
    return np.clip(prediction, 0.5, 5.0)

# Quick test
test_pred = predict_rating_item_cf(1, 1, rating_matrix, item_similarity, K=30)
actual = rating_matrix.loc[1, 1] if 1 in rating_matrix.columns else 'N/A'
print(f"User 1, Movie 1 — Predicted: {test_pred:.2f}, Actual: {actual}")

User 1, Movie 1 — Predicted: 2.62, Actual: 4.0


### Step 4: Generate Top-N Recommendations

In [6]:
def recommend_item_cf(user_id, rating_matrix, item_similarity, movies_df, K=30, top_n=10):
    """
    Recommend top-N movies for a user based on item-based CF predicted ratings.
    """
    user_ratings = rating_matrix.loc[user_id]
    unrated = user_ratings[user_ratings.isna()].index.tolist()
    
    predictions = {}
    for movie_id in unrated:
        pred = predict_rating_item_cf(user_id, movie_id, rating_matrix, item_similarity, K=K)
        if not np.isnan(pred):
            predictions[movie_id] = pred
    
    top_movies = sorted(predictions.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    results = pd.DataFrame({
        'movieId': [m for m, _ in top_movies],
        'Predicted Rating': [round(r, 2) for _, r in top_movies]
    }).merge(movies_df[['movieId', 'title', 'genres']], on='movieId')
    
    return results[['title', 'genres', 'Predicted Rating']]

### Step 5: Test Recommendations

In [7]:
user1_rated = ratings[ratings['userId'] == 1].merge(movies, on='movieId')
print("User 1's top-rated movies:")
print(user1_rated.sort_values('rating', ascending=False)[['title', 'genres', 'rating']].head(10).to_string(index=False))

print("\n" + "="*70)
print("Top 10 Recommendations for User 1 (Item-Based CF, K=30)")
print("="*70)
recommend_item_cf(1, rating_matrix, item_similarity, movies, K=30, top_n=10)

User 1's top-rated movies:
                                 title                             genres  rating
           Seven (a.k.a. Se7en) (1995)                   Mystery|Thriller     5.0
            Usual Suspects, The (1995)             Crime|Mystery|Thriller     5.0
                  Bottle Rocket (1996)     Adventure|Comedy|Crime|Romance     5.0
Dumb & Dumber (Dumb and Dumber) (1994)                   Adventure|Comedy     5.0
                  Billy Madison (1995)                             Comedy     5.0
                      Desperado (1995)             Action|Romance|Western     5.0
                 Canadian Bacon (1995)                         Comedy|War     5.0
                        Rob Roy (1995)           Action|Drama|Romance|War     5.0
                      Pinocchio (1940) Animation|Children|Fantasy|Musical     5.0
                      Tombstone (1993)               Action|Drama|Western     5.0

Top 10 Recommendations for User 1 (Item-Based CF, K=30)


,title,genres,Predicted Rating
0,Two if by Sea (1996),Comedy|Romance,5.0
1,Mr. Wrong (1996),Comedy,5.0
2,"Howling, The (1980)",Horror|Mystery,5.0
3,Booty Call (1997),Comedy|Romance,5.0
4,Selena (1997),Drama|Musical,5.0
5,Run Silent Run Deep (1958),War,5.0
6,Flawless (1999),Drama,5.0
7,Godzilla 2000 (Gojira ni-sen mireniamu) (1999),Action|Adventure|Sci-Fi,5.0
8,Lovely & Amazing (2001),Comedy|Drama|Romance,5.0
9,"Passion of Joan of Arc, The (Passion de Jeanne...",Drama,5.0


In [8]:
print("="*70)
print("Top 10 Recommendations for User 5 (Item-Based CF, K=30)")
print("="*70)
recommend_item_cf(5, rating_matrix, item_similarity, movies, K=30, top_n=10)

Top 10 Recommendations for User 5 (Item-Based CF, K=30)


,title,genres,Predicted Rating
0,Safe (1995),Thriller,5.0
1,Faster Pussycat! Kill! Kill! (1965),Action|Crime|Drama,5.0
2,Naked (1993),Drama,5.0
3,My Own Private Idaho (1991),Drama|Romance,5.0
4,Mighty Joe Young (1998),Action|Adventure|Drama|Fantasy|Thriller,5.0
5,"Out-of-Towners, The (1999)",Comedy,5.0
6,"Draughtsman's Contract, The (1982)",Drama,5.0
7,Birthday Girl (2001),Drama|Romance,5.0
8,Wild at Heart (1990),Crime|Drama|Mystery|Romance|Thriller,5.0
9,Au revoir les enfants (1987),Drama,5.0


### Step 6: Evaluate — RMSE, Precision@K, Recall@K

In [ ]:
def evaluate_item_cf(ratings_df, movies_df, K_neighbors=30, K_rec=10, threshold=4.0, n_test_users=50):
    """
    Evaluate item-based CF with train/test split.
    """
    train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42)
    
    train_matrix = train_df.pivot_table(index='userId', columns='movieId', values='rating')
    train_item_sim = train_matrix.corr(method='pearson', min_periods=5)
    
    # --- RMSE ---
    test_sample = test_df[test_df['userId'].isin(train_matrix.index)].head(500)
    actuals, preds = [], []
    
    for _, row in test_sample.iterrows():
        uid, mid, actual = int(row['userId']), int(row['movieId']), row['rating']
        if mid not in train_item_sim.columns or uid not in train_matrix.index:
            continue
        pred = predict_rating_item_cf(uid, mid, train_matrix, train_item_sim, K=K_neighbors)
        if not np.isnan(pred):
            actuals.append(actual)
            preds.append(pred)
    
    rmse = np.sqrt(mean_squared_error(actuals, preds)) if actuals else float('nan')
    
    # --- Precision@K and Recall@K ---
    precisions, recalls = [], []
    eval_users = test_df['userId'].unique()[:n_test_users]
    
    for uid in eval_users:
        if uid not in train_matrix.index:
            continue
        
        user_test = test_df[(test_df['userId'] == uid) & (test_df['rating'] >= threshold)]
        relevant = set(user_test['movieId'])
        if not relevant:
            continue
        
        recs = recommend_item_cf(uid, train_matrix, train_item_sim, movies_df, K=K_neighbors, top_n=K_rec)
        if recs is None or recs.empty:
            continue
        rec_ids = set(recs.merge(movies_df, left_on='title', right_on='title')['movieId'])
        
        hits = rec_ids & relevant
        precisions.append(len(hits) / K_rec)
        recalls.append(len(hits) / len(relevant))
    
    return {
        'RMSE': rmse,
        'Precision@K': np.mean(precisions) if precisions else 0,
        'Recall@K': np.mean(recalls) if recalls else 0,
        'RMSE_samples': len(actuals),
        'P/R_users': len(precisions)
    }

print("Evaluating Item-Based CF (this may take a minute)...")
results = evaluate_item_cf(ratings, movies, K_neighbors=30, K_rec=10)
print(f"\nRMSE: {results['RMSE']:.4f} (on {results['RMSE_samples']} predictions)")
print(f"Precision@10: {results['Precision@K']:.4f}")
print(f"Recall@10: {results['Recall@K']:.4f}")
print(f"Evaluated users for P/R: {results['P/R_users']}")

Evaluating Item-Based CF (this may take a minute)...


### Step 7: Compare with User-Based CF

In [ ]:
# Side-by-side comparison of recommendations for User 1
print("="*70)
print("COMPARISON: User 1 Recommendations")
print("="*70)

print("\n--- Item-Based CF ---")
item_recs = recommend_item_cf(1, rating_matrix, item_similarity, movies, K=30, top_n=10)
print(item_recs.to_string(index=False))

# Note: To compare with user-based CF, run Task 3 notebook first.
# The key differences are discussed below.

### Step 8: Analysis — Item-Based vs User-Based CF

**Is item-based CF faster or more memory-efficient than user-based CF in the real world? Why?**

Yes, item-based CF is generally more practical in production for several reasons:

1. **Stability**: Item-item similarities are more stable over time. A movie's relationship to other movies doesn't change much as new users join. User-user similarities shift constantly as users rate new items, requiring frequent recomputation.

2. **Scalability**: In most real-world systems, the number of users far exceeds the number of items (e.g., Netflix has ~200M users but ~15K titles). Computing and storing an item-item similarity matrix (items × items) is much smaller than a user-user matrix (users × users).

3. **Pre-computation**: Because item similarities are stable, they can be precomputed offline and cached. User-based CF needs to recompute similarities more frequently as user behavior changes.

4. **Sparsity handling**: Item-based CF tends to handle sparse data better because items accumulate ratings from many users, giving more reliable similarity estimates. User-user overlap can be very sparse.

This is why Amazon famously adopted item-based CF for their recommendation engine — it scales better with growing user bases.